Using Macro F1 as primary metric, along with weighted F1 and Top 3 accuracy 

Also include macro roc auc , macro pr auc, and log loss

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, LabelBinarizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    roc_auc_score,
    top_k_accuracy_score,
    log_loss
)

from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [3]:
def evaluate_model(y_true, y_prob):
    total_classes = y_prob.shape[1]

    # Derive class predictions from highest probability
    y_pred = y_prob.argmax(axis=1)
    
    # Binarize true labels for multiclass PR-AUC calculations
    lb = LabelBinarizer()
    lb.fit(range(total_classes))  # Ensure all classes are considered
    y_true_binarized = lb.transform(y_true)
    
    # Calculate metrics
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Identify classes that contain BOTH positive and negative examples in y_true
    # This prevents Scikit-Learn from crashing if a rare class lacks variance
    valid_auc_classes = (y_true_binarized.sum(axis=0) > 0) & (y_true_binarized.sum(axis=0) < len(y_true))
    
    if np.all(valid_auc_classes):
        macro_prauc = average_precision_score(y_true_binarized, y_prob, average='macro')
        macro_rocauc = roc_auc_score(y_true_binarized, y_prob, multi_class='ovr', average='macro')
    else:
        # Calculate macro metrics only over classes that have valid target distributions
        macro_prauc = average_precision_score(y_true_binarized[:, valid_auc_classes], y_prob[:, valid_auc_classes], average='macro')
        macro_rocauc = roc_auc_score(y_true_binarized[:, valid_auc_classes], y_prob[:, valid_auc_classes], multi_class='ovr', average='macro')
    
    
    top_3_acc = top_k_accuracy_score(y_true, y_prob, k=3, labels=range(total_classes))
    logloss = log_loss(y_true, y_prob, labels=range(total_classes))
    
    return {
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Macro PR-AUC": macro_prauc,
        "Macro ROC AUC": macro_rocauc,
        "Top-3 Accuracy": top_3_acc,
        "Log-Loss": logloss
    }

Read in the datasets

In [4]:
X_train = pd.read_csv('data/scaled_X_train.csv')
X_test = pd.read_csv('data/scaled_X_test.csv')
y_train = pd.read_csv('data/y_train.csv')
y_test = pd.read_csv('data/y_test.csv')

In [6]:
y_train['main_genre'].value_counts()

main_genre
2    769102
9    685703
7    276846
5    244855
0    239686
6    212991
4    181843
8     95952
3     83436
1     69972
Name: count, dtype: int64

In [ ]:
# Electronic     961377 = 2
# Rock           857129 = 9
# Pop            346058 = 7
# Jazz/Blues     306069 = 5
# Classical      299607 = 0
# Metal          266239 = 6
# Hip-Hop/R&B    227304 = 4
# Punk           119940 = 8
# Folk           104295 = 3
# Country         87465 = 1

In [7]:
import pickle
import os

# Compute sample weights dynamically
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

# All models configured to address the class imbalance
models = {
    "Random Forest": RandomForestClassifier(
        class_weight='balanced_subsample', 
        n_jobs=-1, 
        random_state=42
    ),
    
    "LightGBM": lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(set(y_train)) ,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    
    "XGBoost": XGBClassifier(
        objective='multi:softprob', 
        num_class=len(set(y_train)) , 
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ),
    
    "CatBoost": CatBoostClassifier(
        loss_function='MultiClass', 
        auto_class_weights='Balanced',
        random_state=42,
        verbose=50  
    )
}

performance_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    # separate out each model so can view results later
    # added current_model variable to track current model for pickle saving

    if name == "XGBoost":
        xgb_model = model
        xgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
        current_model = xgb_model

    elif name == "LightGBM":
        lgb_model = model
        lgb_model.fit(
            X_train, y_train,
            sample_weight=sample_weights_train, 
            eval_set=[(X_test, y_test)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)] 
        )
        current_model = lgb_model

    elif name == "CatBoost":
        catboost_model = model
        catboost_model.fit(X_train, y_train)
        current_model = catboost_model

    else:
        rf_model = model
        rf_model.fit(X_train, y_train)
        current_model = rf_model

    # Save Model to Pickle Files
    # Cleans up spaces in name for the filename (i.e., "Random Forest" -> "random_forest_model.pkl")
    clean_name = name.lower().replace(' ', '_')
    filepath = os.path.join("models", f"{clean_name}_genre_model.pkl")

    print(f"Saving {name} to {filepath}...")
    with open(filepath, "wb") as file:
        pickle.dump(current_model, file)


    print(f"Evaluating {name} on validation data...")
    if name == "CatBoost":
        y_prob_test = catboost_model.predict_proba(X_test)
    elif name == "LightGBM":
        y_prob_test = lgb_model.predict_proba(X_test)
    elif name == "XGBoost":
        y_prob_test = xgb_model.predict_proba(X_test)
    else:
        y_prob_test = rf_model.predict_proba(X_test)

    # Execute the evaluation function
    metrics = evaluate_model(y_test, y_prob_test)
    performance_results[name] = metrics

print("\n" + "="*50)
print("FINAL MODEL COMPARISON RESULTS")
print("="*50)

df_results = pd.DataFrame(performance_results).T
print(df_results.to_string(formatters={c: '{:,.4f}'.format for c in df_results.columns}))


Training Random Forest...


c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Saving Random Forest to models\random_forest_genre_model.pkl...
Evaluating Random Forest on validation data...

Training LightGBM...


c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\627700\Documents\Music-Analytics-Prediction-RAG\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\627700\Documents

Saving LightGBM to models\lightgbm_genre_model.pkl...
Evaluating LightGBM on validation data...

Training XGBoost...
Saving XGBoost to models\xgboost_genre_model.pkl...
Evaluating XGBoost on validation data...

Training CatBoost...
Learning rate set to 0.118741
0:	learn: 2.1676416	total: 985ms	remaining: 16m 23s
50:	learn: 1.6078352	total: 33.2s	remaining: 10m 16s
100:	learn: 1.5693396	total: 1m 4s	remaining: 9m 33s
150:	learn: 1.5490044	total: 1m 34s	remaining: 8m 52s
200:	learn: 1.5348572	total: 2m 4s	remaining: 8m 15s
250:	learn: 1.5249331	total: 2m 34s	remaining: 7m 42s
300:	learn: 1.5170741	total: 3m 6s	remaining: 7m 13s
350:	learn: 1.5104813	total: 3m 37s	remaining: 6m 43s
400:	learn: 1.5049031	total: 4m 9s	remaining: 6m 13s
450:	learn: 1.5003938	total: 4m 40s	remaining: 5m 41s
500:	learn: 1.4961463	total: 5m 11s	remaining: 5m 10s
550:	learn: 1.4925114	total: 5m 42s	remaining: 4m 39s
600:	learn: 1.4889402	total: 6m 13s	remaining: 4m 7s
650:	learn: 1.4857436	total: 6m 45s	remainin

In [ ]:
# ==================================================
# FINAL MODEL COMPARISON RESULTS
# ==================================================
#               Macro F1 Weighted F1 Macro PR-AUC Macro ROC AUC Top-3 Accuracy Log-Loss
# Random Forest   0.3681      0.4828       0.4114        0.8425         0.8259   1.7087
# LightGBM        0.3779      0.4117       0.4123        0.8498         0.7605   1.6107
# XGBoost         0.3815      0.4160       0.4181        0.8521         0.7643   1.5969
# CatBoost        0.3880      0.4221       0.4246        0.8555         0.7691   1.5823

Confusion matrix for random forest

In [8]:
from sklearn.metrics import confusion_matrix

y_t = y_test
y_p = rf_model.predict(X_test)

matrix = confusion_matrix(y_t, y_p.ravel())

# Define class labels in order
class_labels = ['Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop/R&B', 'Jazz/Blues', 'Metal', 'Pop', 'Punk', 'Rock'] 

# Electronic     961377 = 2
# Rock           857129 = 9
# Pop            346058 = 7
# Jazz/Blues     306069 = 5
# Classical      299607 = 0
# Metal          266239 = 6
# Hip-Hop/R&B    227304 = 4
# Punk           119940 = 8
# Folk           104295 = 3
# Country         87465 = 1

matrix_df = pd.DataFrame(
    matrix, 
    index=[f"Actual {label}" for label in class_labels], 
    columns=[f"Predicted {label}" for label in class_labels]
)

matrix_df

,Predicted Classical,Predicted Country,Predicted Electronic,Predicted Folk,Predicted Hip-Hop/R&B,Predicted Jazz/Blues,Predicted Metal,Predicted Pop,Predicted Punk,Predicted Rock
Actual Classical,42090,75,8667,157,80,3180,299,409,0,4964
Actual Country,701,1795,1096,311,102,1544,17,1459,2,10466
Actual Electronic,6685,143,142682,201,5253,5251,4046,2626,93,25295
Actual Folk,1932,548,3426,879,158,2498,130,1288,4,9996
Actual Hip-Hop/R&B,407,56,21440,40,14940,1051,200,1179,9,6139
Actual Jazz/Blues,6301,316,12913,353,969,24035,199,1649,8,14471
Actual Metal,556,9,7045,15,103,314,25725,146,285,19050
Actual Pop,2212,559,20338,351,2182,3372,534,7528,44,32092
Actual Punk,159,25,3575,19,175,295,3761,165,754,15060
Actual Rock,4724,852,29708,568,1576,6261,11643,4841,593,110660


Save confusion matrix

In [9]:
matrix_df.to_csv('confusion_matrix/genre_rf_confusion_matrix.csv', index=True)

Confusion Matrix for Catboost

In [10]:
from sklearn.metrics import confusion_matrix

y_t = y_test
y_p = catboost_model.predict(X_test)

matrix = confusion_matrix(y_t, y_p.ravel())

# Define class labels in order
class_labels = ['Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop/R&B', 'Jazz/Blues', 'Metal', 'Pop', 'Punk', 'Rock'] 

# Electronic     961377 = 2
# Rock           857129 = 9
# Pop            346058 = 7
# Jazz/Blues     306069 = 5
# Classical      299607 = 0
# Metal          266239 = 6
# Hip-Hop/R&B    227304 = 4
# Punk           119940 = 8
# Folk           104295 = 3
# Country         87465 = 1

matrix_df = pd.DataFrame(
    matrix, 
    index=[f"Actual {label}" for label in class_labels], 
    columns=[f"Predicted {label}" for label in class_labels]
)

matrix_df

,Predicted Classical,Predicted Country,Predicted Electronic,Predicted Folk,Predicted Hip-Hop/R&B,Predicted Jazz/Blues,Predicted Metal,Predicted Pop,Predicted Punk,Predicted Rock
Actual Classical,45452,1131,2813,3425,522,3857,1002,707,284,728
Actual Country,653,9989,236,2378,377,1332,102,1265,316,845
Actual Electronic,12840,3411,89415,7249,30487,11485,12116,13401,5796,6075
Actual Folk,1925,4738,936,6664,840,2379,422,1355,543,1057
Actual Hip-Hop/R&B,615,1299,5424,1666,28976,1966,636,2775,1138,966
Actual Jazz/Blues,6925,5020,3536,5969,3799,29025,742,2754,1050,2394
Actual Metal,1214,308,2730,817,610,597,35564,864,7645,2899
Actual Pop,2748,10966,6881,7393,8009,5103,2005,16477,4069,5561
Actual Punk,283,835,1391,761,1108,554,4669,874,11210,2303
Actual Rock,7089,21354,10223,14375,8229,11803,25424,15005,28549,29375


Save the confusion matrix

In [11]:
matrix_df.to_csv('confusion_matrix/genre_catboost_confusion_matrix.csv', index=True)

Calculate feature importance for random forest

In [13]:
importances = rf_model.feature_importances_

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                           feature_names  importance
4                             onset_rate    0.085919
7                         mfcc_zero_mean    0.085687
3                           danceability    0.082177
5                       average_loudness    0.081447
6                     dynamic_complexity    0.069809
9        tuning_equal_tempered_deviation    0.068571
0                                    bpm    0.066610
12                  mood_aggressive_prob    0.061707
11                       mood_happy_prob    0.061411
8                       tuning_frequency    0.045327
2   bpm_histogram_second_peak_bpm_median    0.043206
1     bpm_histogram_second_peak_bpm_mean    0.043165
16                       mood_aggressive    0.018806
23                    voice_instrumental    0.018263
18                       mood_electronic    0.014765
17                         mood_acoustic    0.013885
21                                timbre    0.011978
20                          voice_gender    0.

In [ ]:
# feature_names  importance
# 4                             onset_rate    0.085919
# 7                         mfcc_zero_mean    0.085687
# 3                           danceability    0.082177
# 5                       average_loudness    0.081447
# 6                     dynamic_complexity    0.069809
# 9        tuning_equal_tempered_deviation    0.068571
# 0                                    bpm    0.066610
# 12                  mood_aggressive_prob    0.061707
# 11                       mood_happy_prob    0.061411
# 8                       tuning_frequency    0.045327
# 2   bpm_histogram_second_peak_bpm_median    0.043206
# 1     bpm_histogram_second_peak_bpm_mean    0.043165
# 16                       mood_aggressive    0.018806
# 23                    voice_instrumental    0.018263
# 18                       mood_electronic    0.014765
# 17                         mood_acoustic    0.013885
# 21                                timbre    0.011978
# 20                          voice_gender    0.010360
# 10                             key_scale    0.009676
# 22                          tonal_atonal    0.009668
# 13                            mood_happy    0.009523
# 14                              mood_sad    0.009091
# 15                          mood_relaxed    0.008543
# 19                            mood_party    0.007954
# 24                             key_key_A    0.007118
# 29                             key_key_D    0.006724
# 27                             key_key_C    0.006594
# 32                             key_key_F    0.006360
# 34                             key_key_G    0.006008
# 25                            key_key_A#    0.005496
# 31                             key_key_E    0.005440
# 30                            key_key_D#    0.003932
# 28                            key_key_C#    0.003824
# 35                            key_key_G#    0.003814
# 26                             key_key_B    0.003646
# 33                            key_key_F#    0.003496

Calculate feature importance for catboost model

In [14]:
importances = catboost_model.feature_importances_

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                           feature_names  importance
7                         mfcc_zero_mean   11.965654
4                             onset_rate   11.771657
5                       average_loudness    7.061485
3                           danceability    6.949206
23                    voice_instrumental    6.611478
16                       mood_aggressive    6.143683
18                       mood_electronic    5.780498
9        tuning_equal_tempered_deviation    5.104567
0                                    bpm    3.990520
8                       tuning_frequency    3.783705
17                         mood_acoustic    3.555240
21                                timbre    3.396220
6                     dynamic_complexity    3.385161
12                  mood_aggressive_prob    3.276941
13                            mood_happy    2.714711
14                              mood_sad    1.953156
11                       mood_happy_prob    1.800792
22                          tonal_atonal    1.

In [ ]:
# feature_names  importance
# 7                         mfcc_zero_mean   11.965654
# 4                             onset_rate   11.771657
# 5                       average_loudness    7.061485
# 3                           danceability    6.949206
# 23                    voice_instrumental    6.611478
# 16                       mood_aggressive    6.143683
# 18                       mood_electronic    5.780498
# 9        tuning_equal_tempered_deviation    5.104567
# 0                                    bpm    3.990520
# 8                       tuning_frequency    3.783705
# 17                         mood_acoustic    3.555240
# 21                                timbre    3.396220
# 6                     dynamic_complexity    3.385161
# 12                  mood_aggressive_prob    3.276941
# 13                            mood_happy    2.714711
# 14                              mood_sad    1.953156
# 11                       mood_happy_prob    1.800792
# 22                          tonal_atonal    1.641655
# 10                             key_scale    1.603916
# 20                          voice_gender    1.417150
# 19                            mood_party    1.010308
# 15                          mood_relaxed    0.786298
# 1     bpm_histogram_second_peak_bpm_mean    0.760220
# 25                            key_key_A#    0.476063
# 31                             key_key_E    0.323364
# 29                             key_key_D    0.315868
# 32                             key_key_F    0.291185
# 26                             key_key_B    0.290793
# 33                            key_key_F#    0.262269
# 24                             key_key_A    0.249856
# 30                            key_key_D#    0.237616
# 35                            key_key_G#    0.237362
# 27                             key_key_C    0.231659
# 28                            key_key_C#    0.215823
# 34                             key_key_G    0.202759
# 2   bpm_histogram_second_peak_bpm_median    0.201161